In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
import importlib.util
import warnings, os

warnings.filterwarnings("ignore")

# Load dataset
CSV_PATH = "92da2bec-b99c-4f8e-91ad-b02d9438d5be.csv"   # UPDATE IF NEEDED
df = pd.read_csv(CSV_PATH)

target = "defaulted" if "defaulted" in df.columns else df.columns[-1]
X = df.drop(columns=[target])
y = df[target]

# Simple preprocessing
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.25, random_state=42, stratify=y
)

# ---------------------------------------
# Train two models & evaluate AUC
# ---------------------------------------
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000)
}

# Check XGBoost availability
if importlib.util.find_spec("xgboost") is not None:
    import xgboost as xgb
    models["XGBoost"] = xgb.XGBClassifier(
        eval_metric="logloss", use_label_encoder=False, random_state=42
    )
else:
    models["GradientBoosting"] = GradientBoostingClassifier()
    print("⚠️ XGBoost not installed—using GradientBoostingClassifier instead.")

results = {}

for name, model in models.items():
    scaler = StandardScaler().fit(X_train)
    X_tr = scaler.transform(X_train)
    X_te = scaler.transform(X_test)

    model.fit(X_tr, y_train)

    y_proba = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_test, y_proba)

    results[name] = (auc, model, scaler)
    print(f"{name} AUC = {auc:.4f}")

# ---------------------------------------
# Select best-performing model
# ---------------------------------------
best_name = max(results, key=lambda k: results[k][0])
best_auc, best_model, best_scaler = results[best_name]

print("\nBest model:", best_name, "| AUC =", best_auc)

X_test_s = best_scaler.transform(X_test)


# ---------------------------------------
# 1. PERMUTATION IMPORTANCE
# ---------------------------------------
perm = permutation_importance(
    best_model, X_test_s, y_test, n_repeats=15, random_state=42
)

perm_df = pd.DataFrame({
    "feature": X.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

# Plot permutation importance
plt.figure(figsize=(7, 6))
plt.barh(perm_df.feature, perm_df.importance)
plt.title(f"Permutation Importance — {best_name}")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("permutation_importance.png")
plt.show()

perm_df.to_csv("permutation_importance.csv", index=False)


# ---------------------------------------
# 2. GLOBAL SHAP ANALYSIS
# ---------------------------------------
if best_name in ["XGBoost", "GradientBoosting"]:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test_s)
else:
    explainer = shap.KernelExplainer(
        best_model.predict_proba, 
        X_train.iloc[:50]  # background sample
    )
    shap_values = explainer.shap_values(X_test_s[:100])

# If classifier returns list → pick positive class
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Mean absolute SHAP values
mean_abs = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    "feature": X.columns,
    "mean_abs_shap": mean_abs
}).sort_values("mean_abs_shap", ascending=False)

# Plot
plt.figure(figsize=(7, 6))
plt.barh(shap_df.feature, shap_df.mean_abs_shap)
plt.title(f"Mean Absolute SHAP Values — {best_name}")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("mean_abs_shap.png")
plt.show()

shap_df.to_csv("mean_abs_shap.csv", index=False)

print("\nFiles generated:")
print(" - permutation_importance.png")
print(" - mean_abs_shap.png")
print(" - permutation_importance.csv")
print(" - mean_abs_shap.csv")
